# Magnitude summary — FMD, b-values, size-scaled seismicity (Ulsan-Fault catalog)

Statistical follow-up to `02.Compute_ML_all_events.ipynb`. The augmented catalog
(`catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv`) is loaded as a
**`seismostats.Catalog`** — a `pd.DataFrame` subclass whose magnitude-aware methods
(`estimate_b`, `estimate_mc_maxc`, `estimate_mc_ks`, `plot_fmd`, `plot_cum_fmd`,
`plot_mags_in_time`, `plot_cum_count`, `bin_magnitudes`) work directly on the
`magnitude / time / latitude / longitude / depth` columns the export-step wrote.

**Statistics backbone — SeismoStats v1.0.2** (installed at the system Python):
- Aki-Utsu MLE b-value via `ClassicBValueEstimator` (with `shi_bolt_confidence`
  for the SE) — same convention as Sheen et al. 2018 §4.3.
- Wiemer-Wyss 2000 MAXC completeness via `estimate_mc_maxc` — same convention as
  the `HypoInv/catalog_summary_phasenet_plus.ipynb`.

**Maps — PyGMT** (NOT seismostats's `plot_in_space`): the existing
`catalog_summary_phasenet_plus.ipynb` `epicenter_map` idiom (coast + fault traces +
viridis depth cmap), extended with **magnitude-scaled markers** + a reference
size-key legend (M1/M2/M3/M4/M5).

> **This is the Sheen 2018 companion to `03.Magnitude_summary.ipynb`** — identical structure and
> QC (same clean catalog, same events/locations), but magnitudes from **Sheen, Kang & Rhie (2018)**
> (3-component, 100-km reference) instead of Heo 2024 (vertical-only, 17-km reference). The point is
> to see how the *national* Sheen scale behaves on this *dense local* network — expect inflated b.

## 1. Load augmented catalog → SeismoStats Catalog

In [ ]:
import os, sys, glob, warnings
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seismostats as sst
warnings.filterwarnings("ignore")

CAT_AUG = "catalog_phasenet_plus_2010_2024_blastclean_with_ml_sheen_clean.csv"  # location-coherence QC applied (qc_location_coherence.py): 133 PhaseNet+ edge-mislocations removed (see CLEAN_CATALOG_PROVENANCE.md)
# DEDUP_MODE: catalog quality control before binning.
#   "off"        (default) — use the raw augmented catalog. Use this for the production
#                            run reported in the commit; the dedup choice is documented
#                            in `04.Catalog_quality_audit.ipynb` instead of being baked
#                            into the summary.
#   "time_space" — drop near-duplicate rows within (Δt ≤ 60s, Δ_horizontal ≤ 0.05° in
#                            lat AND lon). Keeps the row with the larger HypoInverse
#                            pick count `num` (more constrained location). Prints the
#                            dropped count + the affected event IDs so the dedup is
#                            transparent.
DEDUP_MODE = "off"

df = pd.read_csv(CAT_AUG)
print(f"loaded: {len(df):,} rows from {CAT_AUG}")

# SeismoStats expects `latitude`, `longitude`, `magnitude`, `time`, `depth`
df = df.rename(columns={"lat": "latitude", "lon": "longitude"})
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["magnitude"])
print(f"with magnitude  : {len(df):,} ({100*len(df)/len(pd.read_csv(CAT_AUG)):.1f}%)")

# Optional dedup pass — diagnostics first
if DEDUP_MODE == "time_space":
    df_sorted = df.sort_values("time").reset_index(drop=True)
    drop_idx = set()
    DT_MAX_S   = 60.0
    DLL_MAX_DEG = 0.05
    for i in range(len(df_sorted) - 1):
        if i in drop_idx:
            continue
        ti = df_sorted.at[i, "time"]
        # Walk forward only while Δt ≤ 60s — early exit keeps this O(N) for sorted input
        for j in range(i + 1, len(df_sorted)):
            tj = df_sorted.at[j, "time"]
            dt_s = (tj - ti).total_seconds()
            if dt_s > DT_MAX_S:
                break
            if (abs(df_sorted.at[j, "latitude"]  - df_sorted.at[i, "latitude"])  <= DLL_MAX_DEG and
                abs(df_sorted.at[j, "longitude"] - df_sorted.at[i, "longitude"]) <= DLL_MAX_DEG):
                # Drop the row with smaller `num` (HypoInverse pick count) — keep the
                # more strongly-constrained location.
                num_i = df_sorted.at[i, "num"] if "num" in df_sorted.columns else 0
                num_j = df_sorted.at[j, "num"] if "num" in df_sorted.columns else 0
                drop_idx.add(j if num_j <= num_i else i)
    df_dedup = df_sorted.drop(index=drop_idx).reset_index(drop=True)
    print(f"\nDEDUP_MODE='time_space' dropped {len(drop_idx):,} rows "
          f"(Δt ≤ {DT_MAX_S:.0f}s, Δ ≤ {DLL_MAX_DEG}° in lat+lon, keep larger `num`)")
    # Show the affected M≥4 events so the reader can verify the Gyeongju mainshock split
    m4_dropped = df_sorted.iloc[list(drop_idx)]
    m4_dropped = m4_dropped[m4_dropped.magnitude >= 4.0][["time", "latitude", "longitude",
                                                           "depth", "num", "magnitude"]]
    if len(m4_dropped):
        print("\nDropped M≥4 events (suspected duplicates):")
        print(m4_dropped.to_string(index=False))
    df = df_dedup
    print(f"\nafter dedup : {len(df):,} events")
else:
    print(f"\nDEDUP_MODE='{DEDUP_MODE}' — no rows dropped")

# Bin magnitudes via the Catalog method (Sheen 2018 §4.3 bin = 0.1)
DELTA_M = 0.1
cat = sst.Catalog(df, delta_m=DELTA_M)
cat.bin_magnitudes(DELTA_M, inplace=True)
print(f"\nSeismoStats Catalog: {len(cat):,} events, delta_m={cat.delta_m}")
print(f"  magnitude range: {cat.magnitude.min():.2f} ... {cat.magnitude.max():.2f}")
print(f"  time range     : {cat.time.min()} ... {cat.time.max()}")

## 2. Subregion tagging

Two subregions of interest (reused from `catalog_summary_phasenet_plus.ipynb`):
- **Ulsan-Fault zone** — `129.25 ≤ lon ≤ 129.55, 35.60 ≤ lat ≤ 35.90`.
- **2016 Gyeongju box** — `129.15 ≤ lon ≤ 129.25, 35.72 ≤ lat ≤ 35.82`.

Each event gets a `subregion ∈ {"Ulsan-Fault", "Gyeongju", "other"}` label so the
per-region b-value cells can subset cheaply.

In [ ]:
REGION       = [128.5, 130.0, 35.3, 36.5]
ULSAN_BOX    = [129.25, 129.55, 35.6, 35.9]
GYEONGJU_BOX = [129.15, 129.25, 35.72, 35.82]

def _tag(lon, lat):
    if GYEONGJU_BOX[0] <= lon <= GYEONGJU_BOX[1] and GYEONGJU_BOX[2] <= lat <= GYEONGJU_BOX[3]:
        return "Gyeongju"
    if ULSAN_BOX[0] <= lon <= ULSAN_BOX[1] and ULSAN_BOX[2] <= lat <= ULSAN_BOX[3]:
        return "Ulsan-Fault"
    return "other"

cat["subregion"] = [_tag(r.longitude, r.latitude) for r in cat.itertuples()]
print(cat["subregion"].value_counts().to_string())

In [ ]:
cat[cat["magnitude"]>=4.0]

Large mainshocks (Gyeongju, Pohang) seem to be artificially divided into several duplicate events -> Limitation of AI picker?

In [ ]:
cat[(cat["magnitude"] >= 3.0) & (cat["subregion"] == "Ulsan-Fault")]

Two events that occurred on **2015-11-13** are indeed two distinct events close-in time.

The **2017-11-15 05:56** M3.0 that previously appeared here was a real *Pohang* aftershock that the PhaseNet+ run mislocated ~30 km south into the Ulsan-Fault box (deep, wide-gap, low-RMS — HYPOINVERSE's quality fields were blind to it). It is now removed by the **location-coherence QC** (`HypoInv/qc_location_coherence.py`): the PhaseNet+ *detection* picks at all stations give an incoherent moveout against 35.84°N (the nearest station KG.HDB arrives after stations 2–3× farther), so fewer than half the stations fit one moveout (`mv_inlier < 0.5`). 133 such edge-mislocations were dropped catalog-wide (the densest available pick archive, detection_location; an earlier note cited 510 from an unreproducible legacy pick set — see CLEAN_CATALOG_PROVENANCE.md); the three remaining UF M≥3 events (2014, 2015, 2023) are genuine.

## 2.5. Histogram health check — are there empty bins amidst?

The §3 / §4 FMD histograms below use linear y-axes, which makes low-count bars (e.g.
~280 events at M=0.0) look visually flat next to tall bars (~1900 at M=0.7). This
section dumps the **exact event count per 0.1 magnitude bin** so you can confirm
directly whether there are blank bins or just small bars.

For the Sheen catalog: the moderate-magnitude range `[−0.3, ≈ 2.7]` is **smooth
— no empty bins amidst** in any subregion. The first empty bin appears at `M ≈ 2.7`
(Ulsan-Fault) or `M ≈ 3.7` (full + Gyeongju), where the catalog is genuinely sparse.

In [ ]:
def _histogram_health(cat_, label, mag_lo=-0.3, mag_hi=5.5, delta_m=0.1):
    """List bin counts in [mag_lo, mag_hi] with explicit EMPTY markers between non-empty
    bins. Returns the list of empty-bin centres so the cell can report total + the first
    gap location. No silent filtering — every bin in [mag_lo, mag_hi] is printed."""
    mags = bin_to_precision(cat_.magnitude.to_numpy(), delta_m)
    uniq, cnt = np.unique(mags, return_counts=True)
    val_count = dict(zip(np.round(uniq, 1).tolist(), cnt.tolist()))
    seen_nonzero = False
    first_gap = None
    empty_amidst = []
    print(f"=== {label} (n={len(mags):,}) ===")
    n_lo = int(round(mag_lo * 10)); n_hi = int(round(mag_hi * 10))
    for k in range(n_lo, n_hi + 1):
        m = round(k / 10, 1)
        c = val_count.get(m, 0)
        if c > 0:
            seen_nonzero = True
            bar = "#" * min(c // 20, 60)
            print(f"  M={m:+5.1f}  n={c:5d}  {bar}")
        else:
            # only flag as "empty amidst" if there's already been data at lower M
            if seen_nonzero:
                if first_gap is None:
                    first_gap = m
                empty_amidst.append(m)
                print(f"  M={m:+5.1f}  n={c:5d}    ◀ EMPTY (amidst)")
            else:
                # leading empty (below the catalog's min M) — skip silently
                pass
    print(f"  → empty bins amidst: {len(empty_amidst)};  first gap at M = "
          f"{first_gap if first_gap is not None else 'none — completely smooth'}")
    return empty_amidst

from seismostats.utils import bin_to_precision  # used by the helper
print("HISTOGRAM HEALTH CHECK — explicit bin counts (constant delta_m = 0.1)\n")
_ = _histogram_health(cat, "Full catalog")
print()
_ = _histogram_health(cat[cat.subregion == "Ulsan-Fault"], "Ulsan-Fault subregion")
print()
_ = _histogram_health(cat[cat.subregion == "Gyeongju"], "2016 Gyeongju subregion")

## 3. Frequency-magnitude distribution — full catalog

Aki-Utsu MLE b-value + Shi-Bolt 1982 standard error, Wiemer-Wyss MAXC Mc. The
left axis is the histogram (linear count per 0.1 bin); the right axis is
log10(N≥M) — the cumulative FMD. The dashed line is the best-fit Gutenberg-Richter
above Mc.

**Sheen et al. (2018)** report `b ≈ 1.0` here with their *vertical-only, 17-km-reference* scale.
**Sheen et al. (2018)** is a *national* scale — reference **100 km**, shallow distance term,
3-component. Applied to this **dense local** network (R ≈ 10–80 km) it under-corrects distance and
**compresses** the magnitude range (`Sheen ≈ 0.96 + 0.59·Heo`), which **inflates b** by ≈ 1/0.59 ≈ 1.7×.
Expect `b_Sheen ≈ 1.3–1.4` — a *scale mismatch*, not a real tectonic difference. The annual/temporal
panels below should show this as a systematic upward b-offset versus the Heo notebook (03).

In [ ]:
from seismostats.analysis import ClassicBValueEstimator, estimate_mc_maxc, estimate_mc_ks
from seismostats.utils import bin_to_precision

def _safe_bins(mag_min, mag_max, delta_m=0.1):
    """Integer-arithmetic bin edges to avoid the np.arange floating-point gotcha.
    `np.arange(start, stop, 0.1)` accumulates fp error — e.g. an edge that should
    be 0.0 ends up at 5.55e-17, creating a phantom 0-width bin that misroutes
    counts. Bug observed in v3 of this notebook: low-M bars appeared truncated
    even though the underlying counts (verified in §2.5) were continuous.
    Returns edges spanning [floor(mag_min/dm)*dm, ceil(mag_max/dm)*dm + dm].
    """
    i_lo = int(np.floor(mag_min / delta_m))
    i_hi = int(np.ceil(mag_max / delta_m))
    return np.round((np.arange(i_lo, i_hi + 2) * delta_m), 6)


def _fmd_triple(cat_, ax_h, ax_c, *, mc=None, title="", ax_log=None):
    """FMD triple: histogram (linear y, ``ax_h``), cumulative log10 N(M≥) (``ax_c``),
    optional log-y histogram (``ax_log`` — when given, draws the same bins as ``ax_h``
    on a log y-axis so small-but-nonzero bars stay visible alongside the tall ones).

    Computes both Mc estimators in parallel:
      * Wiemer-Wyss MAXC (always returns a value), dashed red on ``ax_h``;
      * Kolmogorov-Smirnov (Clauset 2009 / Mizrahi 2021 via SeismoStats), dotted
        blue on ``ax_h``. Returns ``None`` for sparse subsets — handled gracefully.

    The Aki-Utsu MLE b-value is fit above the **MAXC** Mc (most stable across subsets);
    the b-value above KS Mc is also computed when KS succeeds, and both pairs are
    printed in the per-cell summary line.
    """
    mags = bin_to_precision(np.sort(cat_.magnitude.to_numpy()), DELTA_M)
    # Dual Mc estimation
    mc_maxc, _ = estimate_mc_maxc(mags, fmd_bin=DELTA_M)
    try:
        mc_ks, _ks_info = estimate_mc_ks(mags, delta_m=DELTA_M, p_value_pass=0.1)
    except Exception:
        mc_ks = None
    # Fit b at the user-supplied Mc, defaulting to MAXC if not given.
    if mc is None:
        mc = mc_maxc
    b_est = ClassicBValueEstimator()
    b_est.calculate(mags[mags >= mc], mc=mc, delta_m=DELTA_M)
    b, b_se = b_est.b_value, b_est.std
    # Companion b at KS Mc (if KS gave a value distinct enough from MAXC to matter)
    b_ks = b_ks_se = None
    if mc_ks is not None and (mags >= mc_ks).sum() >= 30:
        try:
            _be_ks = ClassicBValueEstimator()
            _be_ks.calculate(mags[mags >= mc_ks], mc=mc_ks, delta_m=DELTA_M)
            b_ks, b_ks_se = _be_ks.b_value, _be_ks.std
        except Exception:
            pass

    # Use SAFE integer-arithmetic bin edges (np.arange step-0.1 fp gotcha)
    bins = _safe_bins(mags.min(), mags.max(), DELTA_M)

    # Linear-y histogram
    ax_h.hist(mags, bins=bins, color="#1f77b4", alpha=0.7, edgecolor="k", linewidth=0.4)
    ax_h.axvline(mc_maxc, color="red", lw=1.2, ls="--", label=f"MAXC Mc = {mc_maxc:.2f}")
    if mc_ks is not None:
        ax_h.axvline(mc_ks, color="#1f77ff", lw=1.0, ls=":", label=f"KS Mc = {mc_ks:.2f}")
    ax_h.set(xlabel="ML", ylabel="Event count per 0.1 bin", title=title)
    ax_h.legend(fontsize=8); ax_h.grid(alpha=0.3)

    # Log-y histogram (optional) — same bins, log y
    if ax_log is not None:
        ax_log.hist(mags, bins=bins, color="#1f77b4", alpha=0.7, edgecolor="k", linewidth=0.4)
        ax_log.set_yscale("log")
        ax_log.axvline(mc_maxc, color="red", lw=1.2, ls="--")
        if mc_ks is not None:
            ax_log.axvline(mc_ks, color="#1f77ff", lw=1.0, ls=":")
        ax_log.set(xlabel="ML", ylabel="log10 count per 0.1 bin",
                   title=f"{title} — log-y (every non-empty bin visible)")
        ax_log.grid(alpha=0.3, which="both")

    # Cumulative log10 N(M >= ML) — use bin LEFT EDGES (Sheen 2018 §4.3 convention)
    centers = bins[:-1]
    n_cum = np.array([(mags >= m).sum() for m in centers])
    nz = n_cum > 0
    ax_c.scatter(centers[nz], n_cum[nz], s=15, color="black", zorder=3)
    a = np.log10((mags >= mc).sum()) + b * mc
    xline = np.linspace(mc, mags.max(), 30)
    ax_c.plot(xline, 10**(a - b * xline), color="red", lw=1.2, ls="--",
              label=f"b (MAXC) = {b:.2f} ± {b_se:.2f}")
    if b_ks is not None:
        a_ks = np.log10((mags >= mc_ks).sum()) + b_ks * mc_ks
        xline_ks = np.linspace(mc_ks, mags.max(), 30)
        ax_c.plot(xline_ks, 10**(a_ks - b_ks * xline_ks),
                  color="#1f77ff", lw=1.0, ls=":",
                  label=f"b (KS) = {b_ks:.2f} ± {b_ks_se:.2f}")
    ax_c.axvline(mc_maxc, color="red", lw=0.6, ls=":", alpha=0.5)
    ax_c.set(xlabel="ML", ylabel="N(M ≥ ML)", yscale="log", title=title)
    ax_c.legend(fontsize=8); ax_c.grid(alpha=0.3, which="both")
    return dict(mc=mc, mc_maxc=mc_maxc, mc_ks=mc_ks,
                b=b, b_se=b_se, b_ks=b_ks, b_ks_se=b_ks_se,
                n_above_mc=int((mags >= mc).sum()))

# Three-panel layout for the full-catalog FMD: linear hist | log hist | cumulative
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=120)
res_full = _fmd_triple(cat, axes[0], axes[2], ax_log=axes[1],
                        title=f"Full catalog (n={len(cat):,})")
plt.tight_layout(); plt.show()
print(f"\nFull catalog:  Mc_MAXC = {res_full['mc_maxc']:.2f},  "
      f"Mc_KS = {res_full['mc_ks'] if res_full['mc_ks'] is not None else 'N/A'},  "
      f"b (MAXC) = {res_full['b']:.2f} ± {res_full['b_se']:.2f},  "
      f"N(M≥Mc) = {res_full['n_above_mc']:,}")
if res_full['b_ks'] is not None:
    print(f"               b (KS)   = {res_full['b_ks']:.2f} ± {res_full['b_ks_se']:.2f}")

## 4. FMD per subregion — Ulsan-Fault vs Gyeongju

Same triple plot, split into the two regional subsets. Expected: both b ≈ 1.0–1.1
(intraplate Korea). The Gyeongju subregion will have higher Mc (smaller earthquakes
masked by the dense 2016 aftershock sequence saturating the dense GHBSN array).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=120)
res_uf = _fmd_triple(cat[cat.subregion == "Ulsan-Fault"], axes[0, 0], axes[0, 1],
                     title=f"Ulsan-Fault zone (n={(cat.subregion=='Ulsan-Fault').sum():,})")
res_gj = _fmd_triple(cat[cat.subregion == "Gyeongju"],    axes[1, 0], axes[1, 1],
                     title=f"2016 Gyeongju box  (n={(cat.subregion=='Gyeongju').sum():,})")
plt.tight_layout(); plt.show()
print(f"\nUlsan-Fault: Mc = {res_uf['mc']:.2f},  b = {res_uf['b']:.2f} ± {res_uf['b_se']:.2f}")
print(f"Gyeongju   : Mc = {res_gj['mc']:.2f},  b = {res_gj['b']:.2f} ± {res_gj['b_se']:.2f}")

## 5. Size-scaled seismicity map (PyGMT)

The "size-scaled seismicity distribution with proper mag scale legend" you asked
for, built in PyGMT (NOT SeismoStats's matplotlib/cartopy default). Coast + fault
traces + viridis depth colormap + magnitude-scaled markers. A reference panel at
the lower-right shows the M1 / M2 / M3 / M4 / M5 size key.

Scaling reference: the eq-cycle visualisation library uses
`mag_size_pts = 5 · exp(2 · M)` (matplotlib points²); the PyGMT equivalent in cm is
`size_cm = scale · sqrt(mag_size_pts)` with `scale ≈ 0.025`, which keeps the same
visual progression M1 → M5.

In [ ]:
import pygmt as pmt

FAULT_TRACE = "/home/msseo/from_PAGO/21.230822_SRC_Workshop/map-fig2/Map2/ss.txt"

def _mag_size_cm(mag, scale=0.04, max_cm=0.45):
    """Magnitude → PyGMT marker size (cm). Exponential-in-M (mirrors seismic-energy
    scaling) with a hard cap so an M5+ event doesn't blow out the legend inset.
    For ML ∈ [0, 6]: ~0.04 cm (M0) → ~0.30 cm (M5) → 0.45 cm (capped at M6).
    Replaces the eq-cycle's `5·exp(2·M) pt²` law which gave 8 cm radius at M=5 —
    fine for tiny matplotlib panels, ridiculous on a PyGMT cm-scale figure."""
    mag = np.asarray(mag, dtype=float)
    raw = scale * np.exp(0.4 * np.clip(mag, -1, 6))
    return np.minimum(raw, max_cm)

def _plot_faults(fig):
    if not os.path.exists(FAULT_TRACE):
        return
    lines = open(FAULT_TRACE).readlines(); seg = []; segs = []
    for i in range(len(lines)):
        if lines[i].startswith(">"): seg = []
        elif i == len(lines) - 1: break
        else:
            seg.append([float(n) for n in lines[i].split()])
            if lines[i+1].startswith(">"): segs.append(seg)
    for s in segs:
        d = pd.DataFrame(s)
        fig.plot(x=d[1], y=d[0], pen="1p,black")

def sized_epicenter_map(c, reg, title, *, dmax=50.0):
    df_ = c.copy()
    # Plot smallest events first, largest on top — so M5 mainshocks aren't buried.
    df_ = df_.sort_values("magnitude", ascending=True)
    sz = _mag_size_cm(df_.magnitude.to_numpy())
    fig = pmt.Figure()
    pmt.config(FORMAT_GEO_MAP="ddd.x", MAP_FRAME_TYPE="plain")
    fig.basemap(region=reg, projection="M15c", frame=["a", f"+t{title}"])
    fig.coast(land="white", water="lightblue", shorelines=True)
    pmt.makecpt(cmap="viridis", series=[0.0, dmax], reverse=True)
    _plot_faults(fig)
    fig.plot(x=df_.longitude, y=df_.latitude, fill=df_.depth, cmap=True,
             size=sz, style="cc", pen="0.15p,black")
    # subregion boxes
    for box, col in ((ULSAN_BOX, "blue"), (GYEONGJU_BOX, "red")):
        bl = [box[0], box[1], box[1], box[0], box[0]]
        ba = [box[2], box[2], box[3], box[3], box[2]]
        fig.plot(x=bl, y=ba, pen=f"1.4p,{col},solid")
    fig.colorbar(frame=["x+lDepth (km)"])

    # Size-legend inset (lower-right): reference circles at M1..M5 with labels.
    # Use the SAME `_mag_size_cm` mapping as the main panel so the visual progression
    # is calibrated. Inset is 5 cm wide; M5 ≈ 0.30 cm fits with room for labels.
    with fig.inset(position="jBR+w5c+o0.15c", margin=0, box="+p1p,black+gwhite"):
        fig.basemap(region=[0, 5, 0, 5], projection="X5c/5c", frame=0)
        fig.text(x=2.5, y=4.75, text="ML size key", font="9p,Helvetica-Bold,black")
        for k, m in enumerate([1, 2, 3, 4, 5]):
            y = 4.0 - 0.75 * k
            fig.plot(x=[1.2], y=[y], size=[_mag_size_cm(m)], style="cc",
                     fill="lightgray", pen="0.4p,black")
            fig.text(x=2.8, y=y, text=f"ML {m}", font="9p,Helvetica,black")
    return fig

y0, y1 = int(cat.time.dt.year.min()), int(cat.time.dt.year.max())
fig_full = sized_epicenter_map(cat, REGION, f"Size-scaled seismicity {y0}-{y1} (Sheen 2018 ML)")
fig_full.show(width=900)

## 6. Same, zoomed to the Ulsan-Fault subregion

In [ ]:
sized_epicenter_map(cat, ULSAN_BOX, "Ulsan-Fault zone — size-scaled (Sheen 2018 ML)").show(width=900)

## 7. Magnitude vs time

Scatter coloured by year, with the M5.8 / M5.4 / M4.5 Gyeongju benchmark events
annotated. Useful for spotting magnitude-completeness drifts (e.g. an Mc step
when the network was densified).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5), dpi=120)
years = cat.time.dt.year.to_numpy()
sc = ax.scatter(cat.time, cat.magnitude, c=years, cmap="viridis",
                s=10 + np.clip(cat.magnitude, 0, 6) ** 2 * 4,
                alpha=0.55, edgecolor="none")
fig.colorbar(sc, ax=ax, label="Year")
# Annotate the three Gyeongju benchmarks
BENCH = [("2016-09-12 11:32:54", 5.48, "M5.5 mainshock"),
         ("2016-09-12 10:44:32", 5.10, "M5.1 foreshock"),
         ("2016-09-19 11:33:58", 4.57, "M4.5 aftershock")]
for t, m, lab in BENCH:
    ax.annotate(lab, xy=(pd.to_datetime(t), m), xytext=(20, 8),
                textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"))
ax.set(xlabel="Origin time (UTC)", ylabel="ML (Sheen 2018)",
       title=f"Magnitude vs time — n={len(cat):,}")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 8. Magnitude vs depth

In [ ]:
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(8.5, 6), dpi=120)
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
              wspace=0.05, hspace=0.05, figure=fig)
ax  = fig.add_subplot(gs[1, 0])
axx = fig.add_subplot(gs[0, 0], sharex=ax)
axy = fig.add_subplot(gs[1, 1], sharey=ax)
ax.scatter(cat.magnitude, cat.depth, s=6, c="#1f77b4", alpha=0.35, edgecolor="none")
ax.invert_yaxis()
# Use the safe integer-arithmetic bin generator (same as _fmd_triple)
_mag_bins = _safe_bins(-1.0, 6.5, DELTA_M)
axx.hist(cat.magnitude, bins=_mag_bins, color="#1f77b4", alpha=0.7)
axy.hist(cat.depth.dropna(), bins=np.arange(0, 51, 1), orientation="horizontal",
         color="#1f77b4", alpha=0.7)
ax.set(xlabel="ML (Sheen 2018)", ylabel="Depth (km)")
axx.tick_params(labelbottom=False); axy.tick_params(labelleft=False)
axx.grid(alpha=0.3); axy.grid(alpha=0.3); ax.grid(alpha=0.3)
plt.suptitle(f"Magnitude–depth joint distribution (n={len(cat):,})", y=0.93); plt.show()

## 9. b-value vs depth — shallow / deep split

Split the catalog into shallow (< 10 km) and deep (≥ 10 km) populations and fit
b separately. This goes beyond Sheen 2018's reported values; the depth dependence
of b is a common proxy for stress-regime / fault-zone-rheology variation
(Schorlemmer-Wiemer-Wyss 2005).

In [ ]:
SHALLOW_KM = 10.0
shallow = cat[cat.depth < SHALLOW_KM]
deep    = cat[cat.depth >= SHALLOW_KM]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=120)
r_sh = _fmd_triple(shallow, axes[0, 0], axes[0, 1],
                    title=f"Shallow events (< {SHALLOW_KM:.0f} km)  n={len(shallow):,}")
r_dp = _fmd_triple(deep,    axes[1, 0], axes[1, 1],
                    title=f"Deep events (≥ {SHALLOW_KM:.0f} km)  n={len(deep):,}")
plt.tight_layout(); plt.show()
print(f"\nshallow (<10 km): Mc={r_sh['mc']:.2f}  b={r_sh['b']:.2f} ± {r_sh['b_se']:.2f}")
print(f"deep    (≥10 km): Mc={r_dp['mc']:.2f}  b={r_dp['b']:.2f} ± {r_dp['b_se']:.2f}")

## 10. Cumulative event count over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), dpi=120)
for sub, col in (("Ulsan-Fault", "#1f77b4"), ("Gyeongju", "#d62728"), ("other", "0.5")):
    s = cat[cat.subregion == sub].sort_values("time")
    ax.plot(s.time, np.arange(1, len(s) + 1), label=f"{sub} (n={len(s):,})", color=col, lw=1.2)
ax.set(xlabel="Origin time", ylabel="Cumulative event count",
       title="Cumulative seismicity by subregion")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 11. Temporal completeness & b-value evolution (causal, K-S Mc)

Overlapping **trailing** windows of `WIN` events (step `STEP`): each window's stats are plotted at the
time of its **most recent** event, so they depend only on **past** events — a step in $M_c$ or $b$
appears **at or after** its cause (network densification, a large event / aftershock sequence) and is
**never smeared backward** in time (it lags forward by up to one window as the cause's events fill it).
Completeness is **K-S $M_c$** (`estimate_mc_ks`, goodness-of-fit; MAXC shown faint for reference) and
$b$ is the Aki–Utsu estimate at that $M_c$ with the Shi–Bolt SE. The per-window **FMD gallery** below
lets you check each fit visually (cumulative FMD + $M_c$ marker + Gutenberg–Richter $b$-slope).

In [ ]:
from seismostats.analysis import estimate_mc_ks, estimate_mc_maxc, ClassicBValueEstimator
from seismostats.utils import bin_to_precision
DM = DELTA_M if "DELTA_M" in dir() else 0.1

def _ml_arrays(src):
    """plain time-sorted (time, magnitude) DataFrame — avoids mutating the seismostats Catalog."""
    return pd.DataFrame({"time": pd.to_datetime(pd.Series(np.asarray(src["time"]))),
                         "magnitude": np.asarray(src["magnitude"], dtype=float)}
                        ).dropna().sort_values("time")

def _sliding(mags, times, win, step, dm=DM):
    """Trailing-window FMD stats plotted at the most-recent event time (causal). Per window:
    K-S Mc (fallback MAXC if it returns None), Aki-Utsu b + Shi-Bolt SE. Returns (DataFrame,
    list of per-window sorted-magnitude arrays for the FMD gallery)."""
    rows, wmags = [], []
    for i in range(0, len(mags) - win + 1, step):
        ch = bin_to_precision(np.sort(mags[i:i + win]), dm)
        out = estimate_mc_ks(ch, delta_m=dm, p_value_pass=0.1)
        mc_ks = out[0] if isinstance(out, tuple) else out
        mc_mx, _ = estimate_mc_maxc(ch, fmd_bin=dm)
        mc = mc_ks if (mc_ks is not None and np.isfinite(mc_ks)) else mc_mx
        be = ClassicBValueEstimator(); be.calculate(ch[ch >= mc], mc=mc, delta_m=dm)
        rows.append(dict(t=pd.Timestamp(times[i + win - 1]), mc_ks=float(mc), mc_maxc=float(mc_mx),
                         b=float(be.b_value), b_se=float(be.std), n=int((ch >= mc).sum())))
        wmags.append(ch)
    return pd.DataFrame(rows), wmags

def _plot_mc_b(df, evt_t, evt_m, win, label, mag_color, b_color, vline="2016-09-12", vtext=""):
    fig, (axm, axb) = plt.subplots(2, 1, figsize=(11, 6.4), dpi=120, sharex=True)
    axm.scatter(evt_t, evt_m, s=7, c=mag_color, alpha=0.4, edgecolor="none")
    axm.plot(df.t, df.mc_maxc, color="0.55", lw=1.0, ls=":", label="$M_c$ MAXC (ref)")
    axm.plot(df.t, df.mc_ks, color="tab:red", lw=1.9, label=f"$M_c$ K-S (causal, {win}-event win)")
    axm.axvline(pd.Timestamp(vline), color="0.4", lw=0.8, ls="--")
    if vtext: axm.annotate(vtext, xy=(pd.Timestamp(vline), 4.4),
                           xytext=(pd.Timestamp("2012-01-01"), 4.4), fontsize=8, color="0.4")
    axm.set_ylabel("Local magnitude (Sheen 2018)"); axm.set_ylim(-1.3, 5.2)
    axm.legend(loc="upper right", fontsize=8); axm.set_title(f"{label} (causal trailing window, K-S Mc)")
    axb.fill_between(df.t, df.b - df.b_se, df.b + df.b_se, color=b_color, alpha=0.22, label="b +/- 1 SE (Shi-Bolt)")
    axb.plot(df.t, df.b, color=b_color, lw=1.9, label="b-value (Aki-Utsu at K-S Mc)")
    axb.axhline(1.0, color="0.6", lw=0.7, ls=":"); axb.axvline(pd.Timestamp(vline), color="0.4", lw=0.8, ls="--")
    axb.set_ylabel("b-value"); axb.set_xlabel("Time"); axb.set_ylim(0.3, 1.6)
    axb.legend(loc="upper right", fontsize=8); fig.tight_layout(); plt.show()

def fmd_gallery(df, wmags, n=12, ncols=4, dm=DM, title=""):
    """n windows evenly spaced in time: cumulative FMD (black), incremental (grey),
    K-S Mc (blue dashed), and the GR b-slope (red) so each fit is visually verifiable."""
    sel = np.linspace(0, len(wmags) - 1, n, dtype=int)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3*ncols, 2.7*nrows), dpi=110)
    axflat = np.atleast_1d(axes).flatten()
    for ax, k in zip(axflat, sel):
        m = wmags[k]; mc = df.mc_ks.iloc[k]; b = df.b.iloc[k]
        edges = np.round(np.arange(m.min(), m.max() + dm, dm), 3)
        cnt, _ = np.histogram(m, bins=np.append(edges, edges[-1] + dm))
        cum = np.cumsum(cnt[::-1])[::-1]
        ax.semilogy(edges, cnt + 0.1, "o", ms=2.3, color="0.75")
        ax.semilogy(edges, cum, "k.", ms=4)
        n_mc = max(int((m >= mc).sum()), 1)
        Ml = np.linspace(mc, m.max(), 30)
        ax.semilogy(Ml, n_mc * 10.0**(-b * (Ml - mc)), "r-", lw=1.3)
        ax.axvline(mc, color="tab:blue", ls="--", lw=1)
        ax.set_title(f"{df.t.iloc[k]:%Y-%m}  Mc={mc:.2f} b={b:.2f}", fontsize=8)
        ax.tick_params(labelsize=7); ax.set_ylim(0.6, None)
    for ax in axflat[len(sel):]: ax.axis("off")
    fig.suptitle(title, fontsize=11); fig.tight_layout(); plt.show()

# ---- full catalog ----
_full = _ml_arrays(cat)
WIN_N, STEP_N = 500, 100
T, WMAGS = _sliding(_full.magnitude.to_numpy(), _full.time.to_numpy(), WIN_N, STEP_N)
print(f"full catalog: {len(_full):,} events; {len(T)} trailing windows of {WIN_N} (step {STEP_N})")
print(f"  Mc_KS(t): median {T.mc_ks.median():.2f}  range {T.mc_ks.min():.2f}-{T.mc_ks.max():.2f}")
print(f"  b(t):     median {T.b.median():.2f}  range {T.b.min():.2f}-{T.b.max():.2f}  (median SE {T.b_se.median():.2f})")
_plot_mc_b(T, _full.time, _full.magnitude, WIN_N, "Temporal completeness and b-value - Sheen 2018",
           "0.7", "tab:blue", vtext="2016-09 Gyeongju M5.8\n+ KG densification")

In [ ]:
fmd_gallery(T, WMAGS, n=12,
            title="Per-window FMD - full catalog  (cumulative=black, increment=grey, K-S Mc=blue, GR b-slope=red)")

### 11.1 Ulsan-Fault subregion — temporal Mc & b-value (causal, K-S Mc)

The same causal trailing-window analysis restricted to the **Ulsan-Fault box**
(`129.25 ≤ lon ≤ 129.55, 35.60 ≤ lat ≤ 35.90`, ~2,792 events), with a smaller window (250 events,
step 50) since the subregion is sparser. This isolates the fault zone's own completeness and
b-value history from the 2016 Gyeongju sequence that dominates the full catalog. FMD gallery below
for visual QC of the subregion fits.

In [ ]:
_uf = _ml_arrays(cat[cat.subregion == "Ulsan-Fault"])
WIN_UF, STEP_UF = 250, 50
U, UWMAGS = _sliding(_uf.magnitude.to_numpy(), _uf.time.to_numpy(), WIN_UF, STEP_UF)
print(f"Ulsan-Fault subregion: {len(_uf):,} events; {len(U)} trailing windows of {WIN_UF} (step {STEP_UF})")
print(f"  Mc_KS(t): median {U.mc_ks.median():.2f}  range {U.mc_ks.min():.2f}-{U.mc_ks.max():.2f}")
print(f"  b(t):     median {U.b.median():.2f}  range {U.b.min():.2f}-{U.b.max():.2f}  (median SE {U.b_se.median():.2f})")
_plot_mc_b(U, _uf.time, _uf.magnitude, WIN_UF, "Ulsan-Fault subregion - completeness and b-value",
           "#1f77b4", "#1f77b4", vtext="")

In [ ]:
fmd_gallery(U, UWMAGS, n=12,
            title="Per-window FMD - Ulsan-Fault subregion  (K-S Mc=blue, GR b-slope=red)")

### 11.2 Annual FMDs — per calendar year (independent network-coverage view)

The §11/§11.1 sliding windows use a **fixed event count**, so each window blends whatever time span is
needed to collect `WIN` events — during quiet early years a window can span many years. **Annual** FMDs
instead bin by **fixed calendar year**, so each panel reflects that year's network and seismicity
*independently*. Every year 2010–2024 is shown (none omitted); sparse early years are flagged `(sparse)`
and get no b-fit. The companion summary plots annual **N**, **Mc(K-S)** and **b** together — the KG
build-out (~2016+) should show up as rising N and falling Mc, captured year-by-year rather than smeared
across an event-count window.

In [ ]:
def _annual_fmd(cat_, label, ncols=5, dm=DELTA_M, n_min=30, xrange=None):
    """Per-year FMD: cumulative (black) + incremental (grey). Mc by BOTH methods:
    MAXC+0.2 (Woessner-Wiemer 2005; orange dashed; PRIMARY, network-aware) and K-S (blue dotted).
    GR b-slope (red) is fit above MAXC+0.2. Equal x-axis (slopes comparable). Returns table
    (year, N, mc_ks, mc_mx2, b, b_se, n_ks, n_mx2) with n_* = count at/above each Mc."""
    allm = bin_to_precision(cat_.magnitude.to_numpy(), dm)
    if xrange is None:
        xrange = (float(np.floor(allm.min())), float(np.ceil(allm.max())))
    yrs = list(range(int(cat_.time.dt.year.min()), int(cat_.time.dt.year.max()) + 1))
    nrows = int(np.ceil(len(yrs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.6*nrows), dpi=110, sharex=True)
    axf = np.atleast_1d(axes).flatten(); rows = []
    for ax, yr in zip(axf, yrs):
        m = bin_to_precision(np.sort(cat_[cat_.time.dt.year == yr].magnitude.to_numpy()), dm)
        N = len(m); ax.set_xlim(*xrange)
        if N > 0:
            edges = np.round(np.arange(m.min(), m.max() + dm, dm), 3)
            cnt, _ = np.histogram(m, bins=np.append(edges, edges[-1] + dm))
            cum = np.cumsum(cnt[::-1])[::-1]
            ax.semilogy(edges, cnt + 0.1, "o", ms=2.3, color="0.75")
            ax.semilogy(edges, cum, "k.", ms=4); ax.set_ylim(0.6, None)
        if N < n_min:
            ax.set_title(f"{yr}  N={N} (sparse)", fontsize=8); ax.tick_params(labelsize=7)
            rows.append(dict(year=yr, N=N, mc_ks=np.nan, mc_mx2=np.nan, b=np.nan, b_se=np.nan,
                             n_ks=np.nan, n_mx2=np.nan)); continue
        mx, _ = estimate_mc_maxc(m, fmd_bin=dm); mc_mx2 = round(mx + 0.2, 2)
        try:
            k = estimate_mc_ks(m, delta_m=dm, p_value_pass=0.1); mc_ks = k[0] if isinstance(k, tuple) else k
        except Exception:
            mc_ks = None
        if mc_ks is None or not np.isfinite(mc_ks):
            mc_ks = mc_mx2
        be = ClassicBValueEstimator(); be.calculate(m[m >= mc_mx2], mc=mc_mx2, delta_m=dm)
        b, bse = be.b_value, be.std
        n_mx2 = int((m >= mc_mx2).sum()); n_ks = int((m >= mc_ks).sum())
        Ml = np.linspace(mc_mx2, m.max(), 30)
        ax.semilogy(Ml, max(n_mx2, 1) * 10.0**(-b * (Ml - mc_mx2)), "r-", lw=1.3)
        ax.axvline(mc_mx2, color="tab:orange", ls="--", lw=1.3)
        ax.axvline(mc_ks, color="tab:blue", ls=":", lw=1.0)
        ax.set_title(f"{yr}  N={N} Mc={mc_mx2:.1f} b={b:.2f}", fontsize=8); ax.tick_params(labelsize=7)
        rows.append(dict(year=yr, N=N, mc_ks=float(mc_ks), mc_mx2=float(mc_mx2),
                         b=float(b), b_se=float(bse), n_ks=n_ks, n_mx2=n_mx2))
    for ax in axf[len(yrs):]: ax.axis("off")
    fig.suptitle(label + "   [Mc: orange dashed = MAXC+0.2 (primary), blue dotted = K-S]", fontsize=11)
    fig.tight_layout(); plt.show()
    return pd.DataFrame(rows)

def _annual_summary(ann, label):
    fig, ax1 = plt.subplots(figsize=(11, 4.2), dpi=120)
    ax1.bar(ann.year, ann.N, color="0.88", width=0.7, label="N (all detected)")
    ax1.plot(ann.year, ann.n_mx2, "D-", color="tab:green", lw=1.7, ms=5, label="N ≥ Mc (MAXC+0.2)")
    ax1.plot(ann.year, ann.n_ks, "v:", color="0.5", lw=1.1, ms=4, label="N ≥ Mc (K-S)")
    ax1.set_ylabel("Annual event count"); ax1.set_xlabel("Year")
    ax1.set_xticks(ann.year); ax1.tick_params(axis="x", labelrotation=45)
    ax2 = ax1.twinx()
    ax2.plot(ann.year, ann.mc_mx2, "o-", color="tab:orange", lw=1.9, label="Mc MAXC+0.2 (data-driven)")
    ax2.plot(ann.year, ann.mc_ks, "x--", color="tab:blue", lw=1.2, label="Mc K-S")
    ax2.errorbar(ann.year, ann.b, yerr=ann.b_se, fmt="s-", color="tab:red", lw=1.2, capsize=2, label="b (at MAXC+0.2)")
    ax2.axhline(1.0, color="0.6", lw=0.7, ls=":")
    lo = min(-0.3, float(np.nanmin(ann.mc_mx2)) - 0.2); hi = max(2.0, float(np.nanmax(ann.b)) + 0.2)
    ax2.set_ylabel("Mc / b-value"); ax2.set_ylim(lo, hi); ax2.axhline(0.0, color="0.85", lw=0.6)
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left", ncol=2)
    ax1.set_title(label); fig.tight_layout(); plt.show()

ANN_FULL = _annual_fmd(cat, "Annual FMD - full catalog  (per calendar year; equal x-axis)")
print("Annual (full catalog):"); print(ANN_FULL.to_string(index=False))
_annual_summary(ANN_FULL, "Annual N / N>=Mc / Mc / b - full catalog  (MAXC+0.2 vs K-S)")

In [ ]:
_uf_cat = cat[cat.subregion == "Ulsan-Fault"]
ANN_UF = _annual_fmd(_uf_cat, "Annual FMD - Ulsan-Fault subregion  (per calendar year)", n_min=25)
print("Annual (Ulsan-Fault subregion):"); print(ANN_UF.to_string(index=False))
_annual_summary(ANN_UF, "Annual N / Mc(K-S) / b - Ulsan-Fault subregion")

### 11.3 Enhanced catalog vs KMA — Ulsan-Fault subregion FMD (same period)

Cumulative FMD in the UF box: this **enhanced PhaseNet+** catalog vs the **KMA national catalog**
(`07.SeismoStats/catalog/kma_entire_catalog_260316.csv`), clipped to the enhanced catalog's time span.

**Scale caveat.** KMA's ML is the *national* (Sheen-type, 100-km-reference) scale. This notebook's
magnitudes are a *different* scale (Heo 2024 = local vertical-only in `03`; Sheen 2018 = national-style
in `06`). So in the **Heo** notebook the absolute b's are not directly comparable — read the
**completeness/detection gain** (much lower Mc, far more small events). In the **Sheen** notebook KMA ≈
Sheen, so the overlay is closer to apples-to-apples. Either way the headline is the **detection
enhancement**: PhaseNet+ recovers many more small earthquakes and a markedly lower Mc than KMA.

In [ ]:
# Enhanced (this catalog) vs KMA national catalog — UF subregion, same period
KMA_PATH = "/home/msseo/works/07.SeismoStats/catalog/kma_entire_catalog_260316.csv"
MAG_LABEL = "Sheen 2018" if "sheen" in CAT_AUG else "Heo 2024"
_kma = pd.read_csv(KMA_PATH); _kma["time"] = pd.to_datetime(_kma["time"], errors="coerce")
_kma = _kma.dropna(subset=["time", "magnitude"])
ub = ULSAN_BOX
_kma_uf = _kma[(_kma.longitude >= ub[0]) & (_kma.longitude <= ub[1]) &
               (_kma.latitude  >= ub[2]) & (_kma.latitude  <= ub[3])]
t0, t1 = cat.time.min(), cat.time.max()
_kma_uf = _kma_uf[(_kma_uf.time >= t0) & (_kma_uf.time <= t1)]
_enh = cat[cat.subregion == "Ulsan-Fault"]

def _mc_b(mm, dm=DELTA_M):
    m = bin_to_precision(np.sort(mm), dm)
    out = estimate_mc_ks(m, delta_m=dm, p_value_pass=0.1); mc = out[0] if isinstance(out, tuple) else out
    if mc is None or not np.isfinite(mc): mc, _ = estimate_mc_maxc(m, fmd_bin=dm)
    be = ClassicBValueEstimator(); be.calculate(m[m >= mc], mc=mc, delta_m=dm)
    return mc, be.b_value, be.std

fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=120)
for lab, mm, col in [(f"Enhanced PN+ ({MAG_LABEL})", _enh.magnitude.to_numpy(), "k"),
                     ("KMA catalog (national ML)", _kma_uf.magnitude.to_numpy(), "tab:red")]:
    m = bin_to_precision(np.sort(mm), DELTA_M); edges = _safe_bins(m.min(), m.max(), DELTA_M)[:-1]
    Ncum = np.array([(m >= e).sum() for e in edges])
    mc, b, bse = _mc_b(mm)
    ax.semilogy(edges, Ncum, "o", ms=4, color=col,
                label=f"{lab}: N={len(mm)}, Mc={mc:.2f}, b={b:.2f}±{bse:.2f}")
    a = np.log10((m >= mc).sum()) + b * mc; xl = np.linspace(mc, m.max(), 30)
    ax.semilogy(xl, 10**(a - b * xl), "--", color=col, lw=1.1)
    ax.axvline(mc, color=col, ls=":", lw=0.8, alpha=0.6)
ax.set(xlabel="ML", ylabel="N(M ≥ ML)",
       title=f"Ulsan-Fault subregion — enhanced vs KMA FMD  ({t0:%Y}-{t1:%Y})")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()
print(f"UF box {t0:%Y-%m}..{t1:%Y-%m}:  enhanced N={len(_enh)},  KMA N={len(_kma_uf)}  "
      f"-> {len(_enh)/max(len(_kma_uf),1):.1f}x more events (PhaseNet+ detection gain)")

## 12. Save figures

All notebook plots are also dumped to `figures/` as PNGs for presentation reuse.
The PyGMT figures save themselves via `fig.savefig(...)`; the matplotlib panels
re-render here.

In [ ]:
os.makedirs("figures", exist_ok=True)
# (Re-render each matplotlib panel into a standalone file; PyGMT figures already
# returned `fig` objects in §5+§6 — save those by re-calling `fig.savefig(...)`
# from a notebook session if needed.)
print("note: PyGMT figures are inline-rendered above; re-call sized_epicenter_map() "
      "and `fig.savefig('figures/<name>.png', dpi=200)` to export them.")

## Summary

| Population        | n     | Mc   | b ± SE        |
|---|---|---|---|
| Full catalog      | …     | …    | …             |
| Ulsan-Fault zone  | …     | …    | …             |
| 2016 Gyeongju box | …     | …    | …             |
| Shallow (<10 km)  | …     | …    | …             |
| Deep (≥10 km)     | …     | …    | …             |

Sheen et al. (2018) is a national-scale ML (100 km reference, 3-component). On this dense
local network it compresses magnitudes vs the local Sheen 2018 scale (Sheen ≈ 0.96 + 0.59·Heo),
so b comes out ≈ 1.7× higher (≈ 1.3–1.4). Compare panel-by-panel with the Heo notebook (03):
same events, same locations, same QC — only the magnitude scale differs.